# Canine Spleen Benchmark

Import selected canine CT series from Orthanc, run MONAI Label's human `segmentation_spleen` model zero-shot, and compare predictions with same-basename radiologist NIfTI masks.

**Research use only.** The pretrained model was developed on human CT and is not validated for canine clinical use. Keep credentials and all image-derived data outside Git.

## 1. Preflight and secure connection

Start MONAI Label with `docker compose up -d monailabel` and put reference masks in `data/radiologist_masks/<case_id>.nii.gz`. Set `RUN_LIVE = True`, then run the next cell. It securely prompts for the Hostinger Orthanc URL/IP, username, and hidden password at runtime; credentials remain only in notebook memory and are never written to the notebook or repository.

In [ ]:
import shutil
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display
from ipywidgets import Checkbox, Dropdown, FloatSlider, HBox, IntSlider, Output, VBox

from spleen_segmenter.config import Settings
from spleen_segmenter.evaluation import evaluate_case, geometry, geometry_matches, pair_cases
from spleen_segmenter.monailabel import MonaiLabelClient
from spleen_segmenter.orthanc import OrthancClient, import_series

ROOT = Path.cwd().resolve()
RUN_LIVE = False
ORTHANC_VERIFY_TLS = True
MODEL_NAME = "segmentation_spleen"
ALLOW_PREDICTION_RESAMPLE = False
print(f"Project root: {ROOT}")

In [ ]:
settings = None
if RUN_LIVE:
    orthanc_url = input("Hostinger Orthanc URL (for example http://IP:8042): ").strip()
    orthanc_username = input("Orthanc username: ").strip()
    orthanc_password = getpass("Orthanc password (hidden): ")
    monai_label_url = input(
        "MONAI Label URL [http://127.0.0.1:8000]: "
    ).strip() or "http://127.0.0.1:8000"

    if not orthanc_url.startswith(("http://", "https://")):
        raise ValueError("Orthanc URL must begin with http:// or https://")
    if not orthanc_username or not orthanc_password:
        raise ValueError("Orthanc username and password are required")

    settings = Settings(
        root=ROOT,
        orthanc_url=orthanc_url.rstrip("/"),
        orthanc_username=orthanc_username,
        orthanc_password=orthanc_password,
        orthanc_verify_tls=ORTHANC_VERIFY_TLS,
        orthanc_timeout_seconds=120,
        monai_label_url=monai_label_url.rstrip("/"),
        monai_label_timeout_seconds=1800,
        ct_nifti_dir=ROOT / "data" / "ct",
        radiologist_mask_dir=ROOT / "data" / "radiologist_masks",
        prediction_dir=ROOT / "data" / "predictions",
        report_dir=ROOT / "reports",
    )
    del orthanc_password
    settings.ensure_local_directories()
    display(settings.public_summary())  # Credentials are intentionally omitted.
    if shutil.which("dcm2niix") is None:
        raise RuntimeError("dcm2niix is required for Orthanc DICOM conversion")
else:
    print("Dry mode: network, import, and inference cells will be skipped.")

## 2. Discover CT series in Orthanc

`ORTHANC_QUERY` is sent to `/tools/find` together with `Modality=CT`. Adjust it for tags supported by your archive. Returned notebook data is restricted to Orthanc series ID, modality, description, number, and instance count; patient tags are not returned.

In [ ]:
ORTHANC_QUERY = {"SpeciesDescription": "CANINE"}
SEARCH_LIMIT = 100

orthanc = None
series = []
if RUN_LIVE:
    orthanc = OrthancClient(
        settings.orthanc_url,
        username=settings.orthanc_username,
        password=settings.orthanc_password,
        verify_tls=settings.orthanc_verify_tls,
        timeout_seconds=settings.orthanc_timeout_seconds,
    )
    display(orthanc.system_info())
    series = orthanc.find_ct_series(ORTHANC_QUERY, limit=SEARCH_LIMIT)
    display(pd.DataFrame([{
        "orthanc_series_id": item.orthanc_series_id,
        "modality": item.modality,
        "series_description": item.series_description,
        "series_number": item.series_number,
        "instance_count": item.instance_count,
    } for item in series]))

## 3. Select and import series

Map each Orthanc series ID to the exact case basename used by its radiologist mask. Leaving the dictionary empty imports nothing. Temporary archives and DICOM files are deleted after conversion; the local ignored manifest stores only case ID, Orthanc series ID, and CT path.

In [ ]:
SELECTED_SERIES = {
    # "orthanc-series-id": "matching-radiologist-mask-basename",
}

imported = []
import_errors = []
if RUN_LIVE:
    for series_id, case_id in SELECTED_SERIES.items():
        try:
            imported.append(import_series(
                orthanc,
                series_id,
                settings.ct_nifti_dir,
                case_id=case_id,
                manifest_path=ROOT / "data" / "import_manifest.csv",
            ))
        except Exception as exc:
            import_errors.append({"case_id": case_id, "error": str(exc)})
    display(pd.DataFrame([{"case_id": r.case_id, "ct_path": r.ct_path} for r in imported]))
    if import_errors:
        display(pd.DataFrame(import_errors))

## 4. Pair CTs with radiologist masks and validate geometry

CT and reference mask must have the same 3D shape and affine before inference/evaluation. Resolve registration or export errors at the source rather than silently resampling ground truth.

In [ ]:
if RUN_LIVE:
    pairing = pair_cases(settings.ct_nifti_dir, settings.radiologist_mask_dir)
    print(f"Paired cases: {len(pairing.cases)}")
    print(f"CTs missing references: {pairing.missing_references}")
    print(f"Orphan references: {pairing.orphan_references}")
    geometry_rows = []
    valid_cases = []
    for case in pairing.cases:
        ct_image = nib.load(case.ct_path)
        reference_image = nib.load(case.reference_path)
        matches = geometry_matches(ct_image, reference_image)
        geometry_rows.append({
            "case_id": case.case_id,
            "geometry_matches": matches,
            "ct": geometry(ct_image),
            "reference": geometry(reference_image),
        })
        if matches:
            valid_cases.append(case)
    display(pd.DataFrame(geometry_rows))
else:
    pairing = None
    valid_cases = []

## 5. Run zero-shot MONAI Label inference

The client verifies that `segmentation_spleen` is loaded, uploads each geometry-valid CT, and atomically saves the returned NIfTI mask. Existing predictions are not overwritten.

In [ ]:
inference_rows = []
if RUN_LIVE:
    monai = MonaiLabelClient(
        settings.monai_label_url,
        timeout_seconds=settings.monai_label_timeout_seconds,
    )
    model_info = monai.require_model(MODEL_NAME)
    print(f"MONAI Label is ready; model={MODEL_NAME}")
    for case in valid_cases:
        prediction_path = settings.prediction_dir / f"{case.case_id}.nii.gz"
        try:
            if prediction_path.exists():
                status = "existing"
            else:
                monai.infer(case.ct_path, prediction_path, model=MODEL_NAME)
                status = "created"
            inference_rows.append({"case_id": case.case_id, "status": status, "error": ""})
        except Exception as exc:
            inference_rows.append({"case_id": case.case_id, "status": "failed", "error": str(exc)})
    display(pd.DataFrame(inference_rows))

## 6. Evaluate overlap, surfaces, and volumes

Prediction resampling is off by default. If enabled after review, only predictions are resampled to reference geometry with nearest-neighbor interpolation, and the adjustment is recorded per case.

In [ ]:
metric_rows = []
evaluation_errors = []
if RUN_LIVE:
    scored = pair_cases(
        settings.ct_nifti_dir,
        settings.radiologist_mask_dir,
        settings.prediction_dir,
    )
    for case in scored.cases:
        if case.prediction_path is None:
            evaluation_errors.append({"case_id": case.case_id, "error": "prediction missing"})
            continue
        try:
            metrics, reference_geometry, prediction_geometry = evaluate_case(
                case.case_id,
                case.reference_path,
                case.prediction_path,
                allow_resample=ALLOW_PREDICTION_RESAMPLE,
            )
            metric_rows.append(metrics.to_dict())
        except Exception as exc:
            evaluation_errors.append({"case_id": case.case_id, "error": str(exc)})

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
if not metrics_df.empty:
    display(metrics_df.describe(include="all"))
    settings.report_dir.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(settings.report_dir / "case_metrics.csv", index=False)
if evaluation_errors:
    display(pd.DataFrame(evaluation_errors))

## 7. Interactive CT and mask viewer

Use the controls to scroll through axial, coronal, or sagittal CT slices, adjust the CT window, switch cases, and independently show the radiologist and MONAI masks. Radiologist-only voxels are green, MONAI-only voxels are red, and overlap is yellow. Visual review remains essential even when overlap metrics are high.

In [ ]:
def launch_ct_viewer(cases):
    cases_by_id = {case.case_id: case for case in cases}
    if not cases_by_id:
        raise ValueError("No cases with predictions are available for viewing")

    cache = {}

    def load_case(case_id):
        if case_id not in cache:
            case = cases_by_id[case_id]
            ct_image = nib.load(case.ct_path)
            reference_image = nib.load(case.reference_path)
            prediction_image = nib.load(case.prediction_path)
            if not geometry_matches(ct_image, reference_image) or not geometry_matches(
                reference_image, prediction_image
            ):
                raise ValueError(
                    f"Viewer geometry mismatch for {case_id}; resolve it before review"
                )
            cache[case_id] = (
                np.asanyarray(ct_image.dataobj),
                np.asanyarray(reference_image.dataobj) > 0,
                np.asanyarray(prediction_image.dataobj) > 0,
                tuple(float(value) for value in ct_image.header.get_zooms()[:3]),
            )
        return cache[case_id]

    case_control = Dropdown(
        options=sorted(cases_by_id), description="Case:", layout={"width": "420px"}
    )
    plane_control = Dropdown(
        options=("axial", "coronal", "sagittal"), description="Plane:"
    )
    slice_control = IntSlider(
        description="Slice:", min=0, max=1, continuous_update=False
    )
    center_control = FloatSlider(
        description="Center:", min=-1000, max=1000, step=10, value=40,
        continuous_update=False,
    )
    width_control = FloatSlider(
        description="Width:", min=1, max=3000, step=10, value=400,
        continuous_update=False,
    )
    opacity_control = FloatSlider(
        description="Mask opacity:", min=0, max=1, step=0.05, value=0.45,
        continuous_update=False,
    )
    reference_control = Checkbox(value=True, description="Radiologist (green)")
    prediction_control = Checkbox(value=True, description="MONAI (red)")
    viewer_output = Output()

    plane_axis = {"sagittal": 0, "coronal": 1, "axial": 2}

    def update_slice_range(_change=None):
        ct, reference, prediction, _spacing = load_case(case_control.value)
        axis_index = plane_axis[plane_control.value]
        slice_control.max = ct.shape[axis_index] - 1
        visible_mask = reference | prediction
        projection_axes = tuple(i for i in range(3) if i != axis_index)
        areas = visible_mask.sum(axis=projection_axes)
        slice_control.value = int(np.argmax(areas)) if np.any(areas) else ct.shape[axis_index] // 2
        render()

    def render(_change=None):
        ct, reference, prediction, spacing = load_case(case_control.value)
        axis_index = plane_axis[plane_control.value]
        index = min(slice_control.value, ct.shape[axis_index] - 1)
        image_slice = np.take(ct, index, axis=axis_index).T
        reference_slice = np.take(reference, index, axis=axis_index).T
        prediction_slice = np.take(prediction, index, axis=axis_index).T

        overlay = np.zeros((*image_slice.shape, 4), dtype=float)
        opacity = opacity_control.value
        if reference_control.value:
            overlay[reference_slice] = (0.0, 1.0, 0.0, opacity)
        if prediction_control.value:
            overlay[prediction_slice] = (1.0, 0.0, 0.0, opacity)
        if reference_control.value and prediction_control.value:
            overlay[reference_slice & prediction_slice] = (1.0, 1.0, 0.0, opacity)

        displayed_axes = [i for i in range(3) if i != axis_index]
        aspect = spacing[displayed_axes[1]] / spacing[displayed_axes[0]]
        lower = center_control.value - width_control.value / 2
        upper = center_control.value + width_control.value / 2

        viewer_output.clear_output(wait=True)
        with viewer_output:
            figure, axis = plt.subplots(figsize=(7, 7))
            axis.imshow(
                image_slice, cmap="gray", origin="lower", vmin=lower, vmax=upper,
                aspect=aspect,
            )
            axis.imshow(overlay, origin="lower", aspect=aspect)
            axis.set_title(
                f"{case_control.value} — {plane_control.value} slice {index}"
            )
            axis.axis("off")
            plt.show()

    case_control.observe(update_slice_range, names="value")
    plane_control.observe(update_slice_range, names="value")
    slice_control.observe(render, names="value")
    for control in (
        center_control,
        width_control,
        opacity_control,
        reference_control,
        prediction_control,
    ):
        control.observe(render, names="value")

    controls = VBox(
        [
            HBox([case_control, plane_control]),
            slice_control,
            HBox([center_control, width_control]),
            HBox([reference_control, prediction_control, opacity_control]),
        ]
    )
    display(VBox([controls, viewer_output]))
    update_slice_range()
    return controls


if RUN_LIVE and "scored" in globals():
    overlay_cases = [case for case in scored.cases if case.prediction_path is not None]
    if overlay_cases:
        viewer_controls = launch_ct_viewer(overlay_cases)

## Interpretation checklist

- Treat this as external-domain, zero-shot performance of a human-CT model.
- Report scanner/protocol and contrast-phase strata where sample size permits.
- Investigate every geometry failure and visually review every prediction.
- Do not tune thresholds or exclude failures based on test-set radiologist masks.
- Keep de-identification and study approvals outside this public repository.